# PIMMUR multi-agent simulation tutorial

This Colab notebook runs a small **Social Balance** simulation from the PIMMUR multi-agent system (MAS). It uses three agents, two conversation rounds, and one simulation run so you can verify the workflow before increasing the scale.


## 1. Get the code and install dependencies

Run this once in a fresh Colab runtime.

In [ ]:
!git clone https://github.com/JXZhou0224/PIMMUR.git
%cd /content/PIMMUR
%pip install -q -r requirements.txt
%pip install -q -r requirements.txt 'httpx<0.28'
import nltk
nltk.download('vader_lexicon', quiet=True)

Cloning into 'PIMMUR'...
remote: Enumerating objects: 326, done.
remote: Counting objects: 100% (326/326), done.
remote: Compressing objects: 100% (210/210), done.
remote: Total 326 (delta 112), reused 322 (delta 111), pack-reused 0 (from 0)
Receiving objects: 100% (326/326), 7.14 MiB | 7.80 MiB/s, done.
Resolving deltas: 100% (112/112), done.
/content/PIMMUR
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.8/367.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━

True

## 2. Provide an API key

The bundled Social Balance configuration uses the `openai` provider. Please set your API key in `Secrets` on the left of the Colab panel as Name: `OPENAI_API_KEY` and Value: `<your openai api key>` make sure to allow Notebook access


In [ ]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## 3. Inspect an existing configuration

PIMMUR reads a JSON configuration. This tutorial uses the repository's existing Social Balance configuration without changing the simulation code or creating a custom configuration. Review the agent count, number of simulations, and rounds before running it.

In [ ]:
%cd /content/PIMMUR/MAS

from pathlib import Path

config_path = Path('configs/SocialBalance.json')
print(config_path.read_text())

/content/PIMMUR/MAS
{
    "name": "social_balance",
    "agent_n": 3,
    "simulation_n": 64,
    "rounds": 4,
    "provider": "openai",
    "model": "gpt-4o-mini",
    "config_generator":"social_balance"
}


## 4. Run the simulation

`run.py` loads the existing configuration, creates the requested agents, runs the conversation rounds, and writes `result.csv` plus `log.jsonl` under `result/social_balance/`. The bundled configuration requests 64 simulations, so this can use substantial API quota. By running the line below, we can set the simulation count to 5, serving as a minimum demo.

In [ ]:
!sed -i 's/"simulation_n": 64/"simulation_n": 5/' configs/SocialBalance.json

Now we can run the experiment:

In [ ]:
!rm -rf result/social_balance
!mkdir -p result/social_balance # clean the previous results
!python run.py --config configs/SocialBalance.json

Round 1
Generating response with OpenAI model: gpt-4o-mini

You are in a virtual chatroom. Below is a description of yourself.
{"name": "Lamondre", "trait": "A 36-year-old high school history teacher, known for their engaging lessons and ability to connect with students.", "Your_opinion_towards_other_agents": {"Madlyn": "enemy", "Jeffry": "enemy"}}

----

You and others are discussing the following topic:

Throughout the conversation, please express your likes and dislikes towards other agents actively, according to entry in your profile.


----
Never mix up yourself with others.
Here are the history of past conversations:
[]

----



Now, it is your turn to speak.
Please express your opinion and output what you will send to others.

Response: Hey everyone! As a history teacher, I love discussing the past and how it shapes our present. I find it fascinating how different perspectives can lead to diverse interpretations of historical events. 

That said, I have to admit that I don't see

## 5. Inspect the output

`result.csv` contains the final relation state for each completed simulation. `log.jsonl` records the relation trajectory by round.
for easy representation the 0 means friend and 2 means enemy

In [ ]:
import pandas as pd

result_dir = Path('result/social_balance')
display(pd.read_csv(result_dir / 'result.csv'))
print((result_dir / 'log.jsonl').read_text().splitlines()[0])

,n,result
0,0.0,20002


["000000", "000000", "020002", "020002", "020002"]


## 6. Visualize the Social Balance result

Run the repository's existing visualization script from its result directory. It reads `result.csv` and displays the Friends, Enemies, and Mixed case counts.

In [ ]:
import os
import runpy

previous_dir = Path.cwd()
os.chdir(result_dir)
try:
    runpy.run_path('vis.py')
finally:
    os.chdir(previous_dir)

FileNotFoundError: [Errno 2] No such file or directory: '/content/PIMMUR/MAS/result/social_balance/vis.py'

## Next steps

To run another bundled experiment, use `configs/HerdEffect.json` or `configs/NetworkGrowth.json`. To customize a configuration, make a copy, retain a supported `name` such as `social_balance`, and be aware that results are written under `result/<name>/`.